# Batch run RAST on genomes

In [77]:
import os
from tqdm import tqdm
import modelseedpy
from modelseedpy.core.msgenome import MSGenome, MSFeature
from modelseedpy import RastClient
from modelseedpy_ext.re.hash_seq import HashSeq
import pymongo

In [72]:
rast = RastClient()

In [44]:
path = '/scratch/shared/CDM/salterns/data/All_MAGs_proteins'
h_seq = {}
for f in tqdm(os.listdir(path)):
    if f.endswith('.faa'):
        genome = MSGenome.from_fasta(path + '/' + f)
        for f in genome.features:
            seq = f.seq
            if f.seq[-1] == '*':
                seq = f.seq[:-1]
            _hash = HashSeq(seq).hash_value
            if _hash not in h_seq:
                h_seq[_hash] = seq
            else:
                assert h_seq[_hash] == seq

100%|██████████| 3919/3919 [00:52<00:00, 75.30it/s] 


In [95]:
def chunk_dict(data, c_protein_to_rast, chunk_size):
    """Yield successive chunks of size chunk_size from a dictionary."""
    payload = []
    for k, v in data.items():
        doc = c_protein_to_rast.find_one({'_id': k})
        if doc is None:
            payload.append(MSFeature(k, v))
            if len(payload) >= chunk_size:
                yield payload
                payload = []
        else:
            #print(doc)
            pass
    if len(payload) > 0:
        yield payload

In [78]:
mongo = pymongo.MongoClient('mongodb://sequoia.mcs.anl.gov:27017')

In [80]:
mongo_database = mongo['database']

In [97]:
c_protein_to_rast = mongo_database['protein_to_rast']

In [99]:
c_protein_to_rast.estimated_document_count()

2445078

In [96]:
for chunk in chunk_dict(h_seq, c_protein_to_rast, 5000):
    genome = MSGenome()
    genome.features += chunk
    rast.annotate_genome(genome)
    for f in genome.features:
        c_protein_to_rast.insert_one({'_id': f.id, 'value': f.ontology_terms.get('RAST')})

2126923a13a7bcda4e4e680d31275132696d4826584e55b4aa5a388208cee58b {'RAST': ['Voltage-gated chloride channel family protein']}
8730aaa6f28d0d2574ef3b70e4a8342adbd52a56df70a3370f36e06c702d035e {'RAST': ['D-Lactate dehydrogenase, cytochrome c-dependent (EC 1.1.2.4)']}
10c951a42fb14900a014b8c315dfdaf8a319cbb5df4690f88606ea22af9be4c7 {}
